In [ ]:
import os, time, subprocess, pythoncom, psutil, tempfile
import polars as pl
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import HTML, display

# ══════════════════════════════════════════════════════════════════════════════
# PATHS & LOAD
# ══════════════════════════════════════════════════════════════════════════════
first_glob = os.path.expanduser("~").replace("\\", "/")
base_path  = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources"

PROD_PATH = f"{base_path}/Production_Confirmation_Report.parquet"
HC_PATH   = f"{base_path}/ATD_Final.parquet"

print("📂 Loading parquets...")
prod_raw = pl.read_parquet(PROD_PATH)
atd_raw  = pl.read_parquet(HC_PATH)
print(f"✓ Production: {prod_raw.shape[0]:,} rows")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
DISPLAY_NOTEBOOK = True
SEND_EMAIL       = True
ATTACH_RAWDATA   = True
ATTACH_FORMAT    = "xlsx"  # "csv" hoặc "xlsx"

EMAIL_TO = (
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com"
)
 
EMAIL_CC = (
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "van.tran@concentrix.com;"
    "duonghoangvu.pham@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "ExpediaVN_Training_Team@concentrix.com;"
    "ExpediaVN_QA_Team@concentrix.com;"
    "atul.pathak@concentrix.com"
)

OVERRIDE_DATE = None   # None = auto D-1 | "2026-05-25" = specific date
TARGET_CONF   = 90.0

_now  = datetime.now()
days_back = 2 if _now.hour < 6 else 1   # 00:00–05:59 → 2, 06:00+ → 1

if OVERRIDE_DATE:
    report_date = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
    print(f"⚠️  OVERRIDE MODE: {OVERRIDE_DATE}")
else:
    report_date = _now - timedelta(days=days_back)

report_date_s = report_date.strftime("%d-%b-%Y")
report_date_d = report_date.date()
month_start   = report_date.replace(day=1).date()
current_month = report_date.strftime("%y_%m")

def _ordinal(n):
    s = {1:"st",2:"nd",3:"rd"}.get(n%10 if n%100 not in (11,12,13) else 0,"th")
    return f"{n}{s}"

day_ord       = _ordinal(report_date.day)
month_yr      = report_date.strftime("%b'%y")
EMAIL_SUBJECT = f"Expedia VN - Adherence Report - as of the {day_ord} of {month_yr}"
print(f"✓ Subject: {EMAIL_SUBJECT}")
print(f"✓ Period : {month_start} → {report_date_d}")

# ══════════════════════════════════════════════════════════════════════════════
# LOB MAPPING
# ══════════════════════════════════════════════════════════════════════════════
def map_lob(lob):
    if lob is None: return None
    l = lob.lower()
    if "flex"       in l: return None
    if "support_lg" in l: return "Lodging"
    if "support_nl" in l: return "Non_Lodging"
    if "support"    in l: return None
    return lob

# ══════════════════════════════════════════════════════════════════════════════
# HC MAP
# ══════════════════════════════════════════════════════════════════════════════
hc_map = (
    atd_raw
    .select(["Date","Email Id","Supervisor Name","LOB"])
    .unique(subset=["Date","Email Id"])
)

# ══════════════════════════════════════════════════════════════════════════════
# FILTER PRODUCTION DATA
# ══════════════════════════════════════════════════════════════════════════════
prod = (
    prod_raw
    .join(hc_map, on=["Date","Email Id"], how="left")
    .filter(pl.col("Month") == current_month)
    .filter(
        (pl.col("Date") >= pl.lit(month_start)) &
        (pl.col("Date") <= pl.lit(report_date_d))
    )
    .with_columns([
        pl.col("Week_Monday").cast(pl.Utf8).alias("Week_Label"),
        pl.col("LOB")
          .map_elements(map_lob, return_dtype=pl.Utf8)
          .alias("LOB_mapped"),
    ])
    .filter(pl.col("LOB_mapped").is_not_null())
    .with_columns(pl.col("LOB_mapped").alias("LOB"))
)

print(f"✓ MTD rows: {prod.shape[0]:,} | LOBs: {sorted(prod['LOB'].unique().to_list())}")

# ══════════════════════════════════════════════════════════════════════════════
# WEEK LABELS — strictly chronological
# ══════════════════════════════════════════════════════════════════════════════
weeks_sorted = sorted(prod["Week_Label"].unique().to_list())

def wk_lbl(w: str) -> str:
    try: return datetime.strptime(w, "%Y-%m-%d").strftime("%Y-%m-%d")
    except: return w

week_labels = [wk_lbl(w) for w in weeks_sorted]
print(f"✓ Weeks : {week_labels}")

# ══════════════════════════════════════════════════════════════════════════════
# GRAND TOTAL HELPERS
# ══════════════════════════════════════════════════════════════════════════════
total_conf = prod["Confirmed_Production_Mins"].sum()
total_sche = prod["Scheduled_Production_Mins"].sum()
gt_pct_all = round(total_conf / total_sche * 100, 1) if total_sche > 0 else None

# Per-week grand totals
gt_by_week = {}
for w, lbl in zip(weeks_sorted, week_labels):
    sub = prod.filter(pl.col("Week_Label") == w)
    c, s = sub["Confirmed_Production_Mins"].sum(), sub["Scheduled_Production_Mins"].sum()
    gt_by_week[lbl] = round(c/s*100, 1) if s > 0 else None

# ══════════════════════════════════════════════════════════════════════════════
# BUILD PIVOT — manual merge loop (guarantees column order)
# Final columns: [index_col, wk_lbl_0, ..., wk_lbl_N, Grand Total]
# ══════════════════════════════════════════════════════════════════════════════
def build_pivot_df(frame: pl.DataFrame, index_col: str,
                   sort_map: dict = None) -> pd.DataFrame:
    idx_vals = sorted(frame[index_col].drop_nulls().unique().to_list())
    base     = pd.DataFrame({index_col: idx_vals})

    for w, lbl in zip(weeks_sorted, week_labels):
        sub = (
            frame.filter(pl.col("Week_Label") == w)
            .group_by(index_col)
            .agg([
                pl.col("Confirmed_Production_Mins").sum().alias("_c"),
                pl.col("Scheduled_Production_Mins").sum().alias("_s"),
            ])
            .with_columns(
                pl.when(pl.col("_s") > 0)
                  .then((pl.col("_c") / pl.col("_s") * 100).round(1))
                  .otherwise(None).alias(lbl)
            )
            .select([index_col, lbl])
            .to_pandas()
        )
        base = base.merge(sub, on=index_col, how="left")

    grand = (
        frame.group_by(index_col)
        .agg([
            pl.col("Confirmed_Production_Mins").sum().alias("_c"),
            pl.col("Scheduled_Production_Mins").sum().alias("_s"),
        ])
        .with_columns(
            pl.when(pl.col("_s") > 0)
              .then((pl.col("_c") / pl.col("_s") * 100).round(1))
              .otherwise(None).alias("Grand Total")
        )
        .select([index_col, "Grand Total"])
        .to_pandas()
    )
    base = base.merge(grand, on=index_col, how="left")

    if sort_map:
        base["_ord"] = base[index_col].map(sort_map).fillna(99)
        base = base.sort_values("_ord").drop(columns=["_ord"])
    else:
        base = base.sort_values(index_col)

    # Grand Total row
    gt = {index_col: "Grand Total"}
    for lbl in week_labels: gt[lbl] = gt_by_week.get(lbl)
    gt["Grand Total"] = gt_pct_all
    return pd.concat([base.reset_index(drop=True),
                      pd.DataFrame([gt])], ignore_index=True)

LOB_ORDER = {"Lodging":0,"Lodging_Nesting":1,"Non_Lodging":2}
tl_pivot  = build_pivot_df(prod, "Supervisor Name")
lob_pivot = build_pivot_df(prod, "LOB", sort_map=LOB_ORDER)

# Day wise
day_df = (
    prod.group_by("Date")
    .agg([
        pl.col("Confirmed_Production_Mins").sum().alias("_c"),
        pl.col("Scheduled_Production_Mins").sum().alias("_s"),
    ])
    .with_columns(
        pl.when(pl.col("_s") > 0)
          .then((pl.col("_c") / pl.col("_s") * 100).round(1))
          .otherwise(None).alias("Conf_Pct")
    )
    .sort("Date").to_pandas()
)
day_df["Date"] = pd.to_datetime(day_df["Date"]).dt.strftime("%Y-%m-%d")
day_df = pd.concat([day_df,
                    pd.DataFrame([{"Date":"Grand Total","Conf_Pct":gt_pct_all}])],
                   ignore_index=True)

print(f"✓ TL rows: {len(tl_pivot)} | LOB rows: {len(lob_pivot)} | Day rows: {len(day_df)}")
print(f"✓ TL  cols: {tl_pivot.columns.tolist()}")
print(f"✓ LOB cols: {lob_pivot.columns.tolist()}")

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
MET_BG   = "#d4f4e2"; MET_FG  = "#1a5c2a"
MISS_BG  = "#fde8ea"; MISS_FG = "#9b1c2a"
HDR_DARK = "#1a3a5c"; HDR_MID = "#1f5c99"
TOT_BG   = "#1a3a5c"
BLU_ROW  = "#dce8f5"; WHT_ROW = "#ffffff"
BANNER_C = "#8b0020"
FONT     = "font-family:Arial,sans-serif;font-size:11px;"
TH_S     = (f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;"
            f"white-space:nowrap;text-align:center;"
            f"border:1px solid rgba(255,255,255,0.2);")
TD_S     = f"{FONT}padding:4px 8px;border:1px solid #dce8f5;white-space:nowrap;"
TD_TOT   = (f"{FONT}padding:4px 8px;border:1px solid rgba(255,255,255,0.15);"
            f"background:{TOT_BG};color:#fff;font-weight:bold;")

CSS = f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.pw{{overflow-x:auto;margin-bottom:8px}}
.spacer{{height:24px}}
.sec-badge{{display:inline-block;font-size:12px;font-weight:bold;
   background:{HDR_DARK};color:#fff;padding:4px 12px;
   margin:0 0 3px;border-radius:3px}}
.note{{font-size:10.5px;color:#555;background:#f8f8f8;
   border-left:3px solid {HDR_MID};padding:4px 10px;
   margin:0 0 8px;border-radius:0 3px 3px 0}}
.t{{border-collapse:collapse;font-size:11px;font-family:Arial,sans-serif;
   white-space:nowrap;table-layout:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;
   text-align:center;border:1px solid rgba(255,255,255,0.2)}}
.t tbody td{{padding:4px 8px;border:1px solid #dce8f5;text-align:center}}
.t tbody td.lbl{{text-align:left}}
.t tbody tr.blu td{{background:{BLU_ROW}}}
.t tbody tr.wht td{{background:{WHT_ROW}}}
.t tbody tr.tot td{{background:{TOT_BG}!important;color:#fff!important;
   font-weight:bold!important}}
.met{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.miss{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def fv(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return "&#8212;"
    if isinstance(v, float): return f"{v:.1f}%"
    return str(v)

def _color(v, for_email=False):
    if v is None or (isinstance(v, float) and pd.isna(v)): return ("","")
    met = float(v) >= TARGET_CONF
    if for_email:
        s = (f"background:{MET_BG};color:{MET_FG};font-weight:bold;" if met
             else f"background:{MISS_BG};color:{MISS_FG};font-weight:bold;")
        return ("", s)
    return ("met" if met else "miss", "")

def _td_conf(v, is_tot=False, for_email=False, bg_r=WHT_ROW):
    val = fv(v)
    if is_tot:
        return f'<td style="{TD_TOT}text-align:center;">{val}</td>'
    cls, inline = _color(v, for_email)
    if for_email:
        return f'<td style="{TD_S}text-align:center;background:{bg_r};{inline}">{val}</td>'
    return f'<td class="{cls}" style="{TD_S}text-align:center;">{val}</td>'

def _td_lbl(v, is_tot=False, for_email=False, bg_r=WHT_ROW):
    val = str(v) if v is not None else "&#8212;"
    if is_tot:
        return f'<td style="{TD_TOT}text-align:left;">{val}</td>'
    if for_email:
        return f'<td style="{TD_S}text-align:left;background:{bg_r};">{val}</td>'
    return f'<td class="lbl" style="{TD_S}">{val}</td>'

def _sec_hdr(num, title, note, for_email=False):
    badge_s = (f"display:inline-block;font-size:12px;font-weight:bold;"
               f"background:{HDR_DARK};color:#fff;padding:4px 12px;"
               f"margin:0 0 3px;border-radius:3px;")
    note_s  = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
               f"border-left:3px solid {HDR_MID};padding:4px 10px;"
               f"margin:0 0 8px;border-radius:0 3px 3px 0;display:block;")
    if for_email:
        return (f'<p style="margin:20px 0 3px 0;">'
                f'<span style="{badge_s}">{num} {title}</span></p>'
                f'<p style="{note_s}">{note}</p>')
    return (f'<div style="margin:20px 0 3px 0;">'
            f'<span class="sec-badge">{num} {title}</span></div>'
            f'<div class="note">{note}</div>')

# ══════════════════════════════════════════════════════════════════════════════
# PIVOT TABLE RENDERER
# ══════════════════════════════════════════════════════════════════════════════
def build_pivot_table(df: pd.DataFrame, label_col: str,
                      wk_lbls: list, for_email=False) -> str:
    t_cls = "" if for_email else 'class="t" '
    h = [f'<table {t_cls}style="border-collapse:collapse;width:auto;{FONT}">']

    h.append('<thead><tr>')
    h.append(
        f'<th style="{TH_S}background:{HDR_DARK};text-align:left;">'
        f'Conf %<br>'
        f'<span style="font-weight:normal;font-size:10px;opacity:0.85;">'
        f'Row Labels</span></th>'
    )
    for lbl in wk_lbls:
        h.append(f'<th style="{TH_S}background:{HDR_MID};">{lbl}</th>')
    h.append(f'<th style="{TH_S}background:{TOT_BG};">Grand Total</th>')
    h.append('</tr></thead><tbody>')

    for i, row in df.iterrows():
        is_tot = str(row.get(label_col,"")) == "Grand Total"
        even   = i % 2 == 0
        bg_r   = BLU_ROW if even else WHT_ROW
        tr_cls = "tot" if is_tot else ("blu" if even else "wht")

        h.append('<tr>' if for_email else f'<tr class="{tr_cls}">')
        h.append(_td_lbl(row.get(label_col,""), is_tot, for_email, bg_r))

        for lbl in wk_lbls:
            v = row.get(lbl)
            if isinstance(v, float) and pd.isna(v): v = None
            h.append(_td_conf(v, is_tot, for_email, bg_r))

        gt = row.get("Grand Total")
        if isinstance(gt, float) and pd.isna(gt): gt = None
        h.append(_td_conf(gt, is_tot, for_email, bg_r))
        h.append('</tr>')

    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# DAY TABLE RENDERER
# ══════════════════════════════════════════════════════════════════════════════
def build_day_table(df: pd.DataFrame, for_email=False) -> str:
    t_cls = "" if for_email else 'class="t" '
    h = [f'<table {t_cls}style="border-collapse:collapse;width:auto;{FONT}">']
    h.append('<thead><tr>')
    h.append(f'<th style="{TH_S}background:{HDR_DARK};text-align:left;">Row Labels</th>')
    h.append(f'<th style="{TH_S}background:{HDR_MID};">Conf %</th>')
    h.append('</tr></thead><tbody>')

    for i, row in df.iterrows():
        is_tot = str(row.get("Date","")) == "Grand Total"
        even   = i % 2 == 0
        bg_r   = BLU_ROW if even else WHT_ROW
        tr_cls = "tot" if is_tot else ("blu" if even else "wht")
        v      = row.get("Conf_Pct")
        if isinstance(v, float) and pd.isna(v): v = None

        h.append('<tr>' if for_email else f'<tr class="{tr_cls}">')
        h.append(_td_lbl(row.get("Date",""), is_tot, for_email, bg_r))
        h.append(_td_conf(v, is_tot, for_email, bg_r))
        h.append('</tr>')

    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_all(for_email=False):
    parts = []

    # Banner
    bi_s = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    bs_s = f"{FONT}font-size:10.5px;color:#fff;margin:3px 0 0;"
    b_inner = (
        f'<p style="{bi_s}">&#128202; Expedia VN — Adherence Report</p>'
        f'<p style="{bs_s}">As of: <strong>{report_date_s}</strong> &nbsp;|&nbsp; '
        f'Period: {month_start.strftime("%Y-%m-%d")} → {report_date_d} &nbsp;|&nbsp; '
        f'Target: &ge;{TARGET_CONF:.0f}%</p>'
    )
    if for_email:
        parts.append(
            f'<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:0 0 16px;">'
            f'<tr><td style="background:{BANNER_C};padding:10px 14px;border-radius:4px;">'
            f'{b_inner}</td></tr></table>'
        )
    else:
        parts.append(
            f'<div style="background:{BANNER_C};padding:10px 14px;border-radius:4px;'
            f'margin:0 0 16px;">{b_inner}</div>'
        )

    # Section 1
    note1 = (
        f"Conformance % by Team Leader and LOB, pivoted by Week (Week_Monday). "
        f"Period: {month_start.strftime('%Y-%m-%d')} → {report_date_d} &nbsp;|&nbsp; "
        f"&#9632; Green = &ge;{TARGET_CONF:.0f}% &nbsp; &#9632; Red = &lt;{TARGET_CONF:.0f}%"
    )
    parts.append(_sec_hdr("1/", "TL Wise &amp; LOB Wise:", note1, for_email))

    parts.append('<div class="pw">' if not for_email else "")
    parts.append(build_pivot_table(tl_pivot,  "Supervisor Name", week_labels, for_email))
    parts.append('</div>' if not for_email else "")

    if for_email:
        parts.append('<table width="100%" border="0" cellspacing="0" cellpadding="0">'
                     '<tr><td style="height:12px;font-size:1px;">&nbsp;</td></tr></table>')
    else:
        parts.append('<div style="margin-top:12px;"></div>')

    parts.append('<div class="pw">' if not for_email else "")
    parts.append(build_pivot_table(lob_pivot, "LOB", week_labels, for_email))
    parts.append('</div>' if not for_email else "")

    # Section 2
    if for_email:
        parts.append('<table width="100%" border="0" cellspacing="0" cellpadding="0">'
                     '<tr><td style="height:24px;font-size:1px;">&nbsp;</td></tr></table>')
    else:
        parts.append('<div class="spacer"></div>')

    note2 = (
        f"Daily Conformance % from {month_start.strftime('%Y-%m-%d')} "
        f"to {report_date_d}. Sorted by date ascending. "
        f"&#9632; Green = &ge;{TARGET_CONF:.0f}% &nbsp; &#9632; Red = &lt;{TARGET_CONF:.0f}%"
    )
    parts.append(_sec_hdr("2/", "Day Wise (MTD):", note2, for_email))

    parts.append('<div class="pw">' if not for_email else "")
    parts.append(build_day_table(day_df, for_email))
    parts.append('</div>' if not for_email else "")

    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def build_greeting():
    month_long = report_date.strftime("%B %Y")
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    I would like to share the Adherence Report as of the {day_ord} of {month_long}
    for Expedia VN.
</p>
<p style="{FONT}font-size:12px;margin:0 0 4px;line-height:1.7">
    *This one will replace Conformance Report and target at {TARGET_CONF:.0f}%.
</p>
<p style="{FONT}font-size:12px;margin:0 0 16px;line-height:1.7">
    ** Adherence based on IEX open time and actual productivity delivered
    on assigned interval capped by open time.
</p>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 14px;">
"""

def build_signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:14px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;line-height:1.7">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;">
    Analyst, WFM Real Time Management</p>
<br>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:{HDR_MID};font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:10px;">
    Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')} &nbsp;|&nbsp;
    Source: Production_Confirmation_Report.parquet
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# BUILD RAWDATA ATTACHMENT
# ══════════════════════════════════════════════════════════════════════════════
def build_rawdata_attachment(frame: pl.DataFrame, fmt: str = "xlsx") -> str:
    """
    Export prod DataFrame sang CSV hoặc XLSX multi-sheet.
    Trả về đường dẫn file tạm để attach vào email.
    """
    df_export = frame.to_pandas()
    fname = f"Adherence_Rawdata_{report_date.strftime('%Y%m%d')}.{fmt}"
    tmp_path = os.path.join(tempfile.gettempdir(), fname)

    if fmt == "csv":
        # utf-8-sig để Excel mở đúng, không lỗi font
        df_export.to_csv(tmp_path, index=False, encoding="utf-8-sig")

    elif fmt == "xlsx":
        with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
            # Sheet 1: Raw Data
            df_export.to_excel(writer, index=False, sheet_name="Raw Data")
            ws_raw = writer.sheets["Raw Data"]
            for col in ws_raw.columns:
                max_len = max(len(str(cell.value or "")) for cell in col)
                ws_raw.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

            # Sheet 2: TL Pivot
            tl_pivot.to_excel(writer, index=False, sheet_name="TL Pivot")
            ws_tl = writer.sheets["TL Pivot"]
            for col in ws_tl.columns:
                max_len = max(len(str(cell.value or "")) for cell in col)
                ws_tl.column_dimensions[col[0].column_letter].width = min(max_len + 2, 30)

            # Sheet 3: LOB Pivot
            lob_pivot.to_excel(writer, index=False, sheet_name="LOB Pivot")
            ws_lob = writer.sheets["LOB Pivot"]
            for col in ws_lob.columns:
                max_len = max(len(str(cell.value or "")) for cell in col)
                ws_lob.column_dimensions[col[0].column_letter].width = min(max_len + 2, 30)

            # Sheet 4: Day Wise
            day_df.to_excel(writer, index=False, sheet_name="Day Wise")

    else:
        raise ValueError(f"Unsupported format: {fmt}. Use 'csv' or 'xlsx'.")

    size_kb = os.path.getsize(tmp_path) / 1024
    print(f"✓ Attachment ready: {fname} ({size_kb:.1f} KB) → {tmp_path}")
    return tmp_path

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY NOTEBOOK
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb_html = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<style>{CSS}</style></head><body>"
        + build_all(for_email=False)
        + "</body></html>"
    )
    escaped = nb_html.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{escaped}" style="width:100%;border:none;min-height:800px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'
    ))
    print("✓ Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client

    email_html = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + build_greeting()
        + build_all(for_email=True)
        + build_signature()
        + "</div>"
    )

    def send_auto(to, cc, subject, html_body, attachments=None, quit_after=True):
        """
        attachments: list of absolute file paths để đính kèm.
        """
        pythoncom.CoInitialize()
        was_on = any(p.name().lower() == "outlook.exe" for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("⏳ Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol = win32com.client.Dispatch("Outlook.Application")
            ol.GetNamespace("MAPI").Logon()
            mail = ol.CreateItem(0)
            mail.To      = to
            mail.CC      = cc
            mail.Subject = subject
            mail.HTMLBody = html_body

            # ── Attach files ───────────────────────────────────────────────
            if attachments:
                for fpath in attachments:
                    abs_path = os.path.abspath(fpath)
                    if os.path.exists(abs_path):
                        mail.Attachments.Add(abs_path)
                        print(f"✓ Attached: {os.path.basename(abs_path)}")
                    else:
                        print(f"⚠️  Not found, skipped: {abs_path}")

            mail.Send()
            print(f"✓ Email sent → {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("✓ Outlook closed")
                except: pass

    attachments = []
    if ATTACH_RAWDATA:
        attach_path = build_rawdata_attachment(prod, fmt=ATTACH_FORMAT)
        attachments.append(attach_path)

    # Send email
    send_auto(
        EMAIL_TO, EMAIL_CC,
        EMAIL_SUBJECT, email_html,
        attachments=attachments,
        quit_after=True,
    )

    for f in attachments:
        try: os.remove(f); print(f"✓ Cleaned: {f}")
        except: pass

📂 Loading parquets...
✓ Production: 259,508 rows
✓ Subject: Expedia VN - Adherence Report - as of the 24th of Aug'26
✓ Period : 2026-08-01 → 2026-08-24
✓ MTD rows: 26,876 | LOBs: ['Lodging', 'Non_Lodging']
✓ Weeks : ['2026-07-27', '2026-08-03', '2026-08-10', '2026-08-17', '2026-08-24']
✓ TL rows: 8 | LOB rows: 3 | Day rows: 25
✓ TL  cols: ['Supervisor Name', '2026-07-27', '2026-08-03', '2026-08-10', '2026-08-17', '2026-08-24', 'Grand Total']
✓ LOB cols: ['LOB', '2026-07-27', '2026-08-03', '2026-08-10', '2026-08-17', '2026-08-24', 'Grand Total']


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Conf %Row Labels,2026-07-27,2026-08-03,2026-08-10,2026-08-17,2026-08-24,Grand Total
Ann,93.2%,92.6%,91.8%,94.2%,94.3%,92.9%
Chau Thien Kim,90.0%,90.1%,85.4%,92.4%,86.7%,90.5%
Mia Minh Le,&#8212;,93.0%,91.8%,93.7%,&#8212;,92.7%
Nguyen Thi Anh Thu,92.5%,90.8%,91.0%,&#8212;,86.9%,90.8%
Tran Hoang My Anh,92.0%,90.1%,89.8%,90.8%,92.4%,90.5%
Tran Thao Uyen,90.2%,91.2%,92.0%,92.6%,89.5%,91.7%
Truong Thien Thanh Toan,89.8%,90.1%,89.9%,92.7%,92.8%,90.8%
Grand Total,91.1%,90.9%,90.8%,92.6%,89.3%,91.2%
Conf %Row Labels,2026-07-27,2026-08-03,2026-08-10,2026-08-17,2026-08-24,Grand Total
Lodging,91.6%,90.9%,90.8%,92.6%,89.3%,91.4%


✓ Display done
✓ Attachment ready: Adherence_Rawdata_20260824.xlsx (2717.4 KB) → C:\Users\HUUCHI~1.NGU\AppData\Local\Temp\Adherence_Rawdata_20260824.xlsx
✓ Attached: Adherence_Rawdata_20260824.xlsx
✓ Email sent → puneet.suneja@concentrix.com;kirpan.patar@concentrix.com;ML.HOC.Expedia.Hierarchy@concentrix.com
✓ Cleaned: C:\Users\HUUCHI~1.NGU\AppData\Local\Temp\Adherence_Rawdata_20260824.xlsx
